# Importation des librairies

In [1]:
import os

In [2]:
os.environ["KERAS_BACKEND"]="torch"
import torch
import gc
import json
import collections
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import numpy as np
from PIL import Image
import tensorflow as tf
import keras
from keras import layers
from keras.applications.inception_v3 import InceptionV3
from sklearn.utils import shuffle

# Chemins et imports
Le notebook sélectionne le backend Torch avant d’importer Keras. Il cherche le dossier models à partir du répertoire de travail, puis définit les chemins des modèles et des données.

In [3]:
PROJECT_DIR = os.path.abspath(os.getcwd())
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CLASSIFIER_WEIGHTS = os.path.join(MODELS_DIR, "model_full.weights.h5")
DENOISER_MODEL = os.path.join(MODELS_DIR, "unet_denoiser_livrable2.keras")
DATASETS_DIR = os.path.join(PROJECT_DIR, "datasets", "Livrable 3")
COCO_DIR = os.path.join(DATASETS_DIR, "train2014")
PHOTOS_DIR = os.path.join(DATASETS_DIR, "photos")
DENOISED_DIR = os.path.join(DATASETS_DIR, "denoised_images")
ANNOTATIONS_DIR = os.path.join(DATASETS_DIR, "annotations")
ANNOTATION_FILE = os.path.join(ANNOTATIONS_DIR, "captions_train2014.json")

# Etape 1 : classification Photo / Non-photo
Le modèle du livrable 1 reconnaît cinq classes : Painting, Photo, Schematics, Sketch et Text. Il faut donc garder ses cinq sorties : l’étape est dite binaire parce qu’on transforme ensuite sa prédiction en deux groupes, Photo et Non-photo.

Pour chaque image, le code :

la redimensionne en 128 × 128 ;
la passe au modèle, qui divise ses pixels par 255 grâce à sa couche Rescaling ;
récupère la probabilité associée à Photo ;
la compare au seuil 0,2301, défini dans le livrable 1.
Si la probabilité atteint le seuil, l’image est considérée comme une photo. Sinon, elle reste dans son dossier d’origine. Le code déplace les photos retenues dans datasets/photos — il ne les copie pas — mais ne déplace pas et ne supprime pas les autres images.

In [4]:
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
NUM_CLASSES = len(CLASS_NAMES)
PHOTO_CLASS_INDEX = CLASS_NAMES.index("Photo")
THRESHOLD = 0.2301

In [5]:
def load_classifier_model():
    model = keras.models.load_model(CLASSIFIER_WEIGHTS, compile=False)
    return model

def load_image_as_array(path, target_size=(128, 128)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32)

def classify_images(image_dir: str | Path,classifier: keras.models.Model):
    results = []
    image_dir = Path(image_dir)

    for image_file in tqdm(image_dir.iterdir()):
        image_array = load_image_as_array(str(image_file))
        image_array = np.expand_dims(image_array, axis=0)

        probabilities = classifier.predict(image_array,verbose=0)
        proba_photo = float(probabilities[0][PHOTO_CLASS_INDEX])

        results.append((str(image_file),proba_photo,proba_photo >= THRESHOLD))

    return results

def move_photos_to_folder(classification_results, dest_dir=PHOTOS_DIR):
    os.makedirs(dest_dir, exist_ok=True)
    moved = []
    for path, _, is_photo in classification_results:
        if is_photo:
            if os.path.abspath(os.path.dirname(path)) != os.path.abspath(dest_dir):
                filename = os.path.basename(path)
                dest_path = os.path.join(dest_dir, filename)
                if os.path.exists(dest_path):
                    base, extension = os.path.splitext(filename)
                    suffix = 1
                    while os.path.exists(dest_path):
                        dest_path = os.path.join(dest_dir, f"{base}_{suffix}{extension}")
                        suffix += 1
                shutil.move(path, dest_path)
                path = dest_path
            moved.append(path)
    return moved

# Etape 2 : débruitage avec le U-Net du livrable 2
Chaque photo retenue est chargée en RGB, redimensionnée en 299 × 299 et convertie en valeurs entre 0 et 1. Le U-Net du livrable 2 est ensuite appliqué à l’image, et le résultat est sauvegardé dans datasets/denoised_images.

Le U-Net effectue des opérations de pooling qui nécessitent des dimensions compatibles avec des réductions par deux. Le code complète donc temporairement les dimensions de 299 à 300, puis recadre le résultat à 299 × 299 avant de le sauvegarder.

In [6]:
def load_denoiser_model():
    denoiser = keras.models.load_model(DENOISER_MODEL, compile=False)
    return denoiser

def load_image_for_denoiser(path, target_size=(299, 299)):
    with Image.open(path) as image:
        image = image.convert("RGB").resize(target_size)
        return np.asarray(image, dtype=np.float32) / 255.0

def denoise_images(image_paths, denoiser, output_dir=DENOISED_DIR):
    os.makedirs(output_dir, exist_ok=True)

    for path in tqdm(image_paths):
        arr = load_image_for_denoiser(path)
        height, width = arr.shape[:2]
        pad_height = (-height) % 4
        pad_width = (-width) % 4
        padded = np.pad(arr, ((0, pad_height), (0, pad_width), (0, 0)), mode="edge")
        batch = np.expand_dims(padded, axis=0)
        denoised = keras.ops.convert_to_numpy(denoiser(batch, training=False))[0]
        denoised = denoised[:height, :width, :]
        denoised = np.clip(denoised, 0.0, 1.0)

        denoised_img = Image.fromarray(np.rint(denoised * 255).astype("uint8"))
        filename = os.path.basename(path)
        out_path = os.path.join(output_dir, filename)
        denoised_img.save(out_path)
        gc.collect()
        torch.cuda.empty_cache()


# Etape 3: Chargement des données

## Création des vecteurs captions et images

In [7]:
def create_captions_and_path_vectors(annotation_file):
    image_path_to_caption = collections.defaultdict(list)
    train_captions = []
    img_name_vector = []
    mark_start, mark_end = '<start> ', ' <end>'

    with open(annotation_file, 'r') as f:
        annotations = json.load(f)

    for val in annotations['annotations']:
        caption =  mark_start + val['caption'].strip() + mark_end
        image_path = os.path.join(DENOISED_DIR,'COCO_train2014_' + '%012d.jpg' % (val['image_id']))

        if not os.path.isfile(image_path):
            continue
        image_path_to_caption[image_path].append(caption)
        
    train_image_paths = list(image_path_to_caption.keys())


    for image_path in train_image_paths:
        caption_list = image_path_to_caption[image_path]
        train_captions.extend(caption_list)
        [img_name_vector.append(image_path) for _ in range(len(caption_list))]

    return img_name_vector, train_captions

## Prétraitement des annotations

In [8]:
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

def annotation_preprocessing(train_captions, top_k=5000):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,
                                                    oov_token="<unk>",
                                                    filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~')

    tokenizer.fit_on_texts(train_captions)
    tokenizer.word_index['<pad>'] = 0
    tokenizer.index_word[0] = '<pad>'
    train_seqs = tokenizer.texts_to_sequences(train_captions)
    cap_vector = keras.preprocessing.sequence.pad_sequences(train_seqs, padding='post')
    max_length = calc_max_length(train_seqs)

    return cap_vector, max_length

# Etape 4: Extraction des caractéristiques avec InceptionV3

In [9]:
def build_inception_feature_extractor():
    # Charge InceptionV3 pré-entraîné sur ImageNet, sans la tête de classification
    image_model = InceptionV3(weights="imagenet", include_top=False)
    new_input = keras.Input(shape = (299, 299, 3))
    hidden_layer = image_model(new_input)

    return keras.Model(new_input, hidden_layer)

def load_image_for_inception(path, target_size=(299, 299)):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = (img / 127.5) - 1.0
    
    return img, path

def extract_and_save_inception_features(img_name_vector, feature_extractor, batch_size=8):
    encode_train = sorted(set(img_name_vector))

    image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
    image_dataset = image_dataset.map(
    load_image_for_inception, num_parallel_calls=1).batch(batch_size)

    with torch.inference_mode():
        for img, path in tqdm(image_dataset):
            batch_features = feature_extractor(img, training=False)
            batch_features = keras.ops.reshape(batch_features,
                                        (batch_features.shape[0], -1, batch_features.shape[3]))
            for bf, p in zip(batch_features, path):
                bactch_feature = keras.ops.convert_to_numpy(bf)
                path_of_feature = p.numpy().decode("utf-8")
                np.save(path_of_feature, bactch_feature)

            del img
            del path
            del batch_features

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# Etape 5 : séparation train / validation (80/20)

In [10]:
def map_func(img_name, cap):
    img_tensor = np.load(img_name.decode('utf-8')+'.npy')
    return img_tensor, cap

def data_splitting(img_name_vector, cap_vector, buffer_size, batch_size):

    img_to_cap_vector = collections.defaultdict(list)
    for img, cap in zip(img_name_vector, cap_vector):
        img_to_cap_vector[img].append(cap)

    img_keys = list(img_to_cap_vector.keys())
    img_keys_shuffled = shuffle(img_keys)
    slice_index = int(len(img_keys)*0.8)
    img_name_train_keys, img_name_val_keys = img_keys_shuffled[:slice_index], img_keys_shuffled[slice_index:]

    # boucle pour le jeu d'entrainement
    img_name_train = []
    cap_train = []
    for imgt in img_name_train_keys:
        capt_len = len(img_to_cap_vector[imgt])
        img_name_train.extend([imgt] * capt_len)
        cap_train.extend(img_to_cap_vector[imgt])

    # Boucle pour le jeu de test
    img_name_val = []
    cap_val = []
    for imgv in img_name_val_keys:
        capv_len = len(img_to_cap_vector[imgv])
        img_name_val.extend([imgv] * capv_len)
        cap_val.extend(img_to_cap_vector[imgv])

    dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))
    dataset = dataset.map(lambda item1, item2: tf.numpy_function(
            map_func, [item1, item2], [tf.float32, tf.int32]),
            num_parallel_calls=tf.data.experimental.AUTOTUNE)

    # Melanger les donnees et les diviser en batchs
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

    return dataset

# Etape 5 : pipeline complète

In [11]:
photos_path = Path(PHOTOS_DIR)
photos_moved = [os.path.join(photos_path, f.name) for f in photos_path.iterdir() if f.is_file()]

In [ ]:
def run_pipeline():
    print(f"1. Classification des images")
    classifier = load_classifier_model()
    classification_results = classify_images(COCO_DIR, classifier)
    photos_moved = move_photos_to_folder(classification_results)
    print(f"{len(photos_moved)} photos deplacees dans {PHOTOS_DIR}.")
    print("2. Débruitage")
    print("3. Chargement des donnees et pretraitement") 
    img_name_vector, train_captions = create_captions_and_path_vectors(annotation_file=ANNOTATION_FILE)
    cap_vector, max_length = annotation_preprocessing(train_captions)

    print("4. Extraction des caracteristiques avec InceptionV3") 
    feature_extractor = build_inception_feature_extractor()
    extract_and_save_inception_features(img_name_vector, feature_extractor)
    print(f"Extraction terminee dans : {DENOISED_DIR}")

    print("5. Séparation train/validation/test")
    dataset = data_splitting(img_name_vector, cap_vector, buffer_size=1000, batch_size=64)

    return dataset, cap_vector, max_length, train_captions


In [13]:
run_pipeline()

3. Chargement des donnees et pretraitement
4. Extraction des caracteristiques avec InceptionV3


  0%|          | 0/9455 [00:00<?, ?it/s]

Extraction terminee dans : c:\Users\brago\Documents\# Daryl goat\CESI\Leyenda\datasets\Livrable 3\denoised_images
